# PlayMatch on Colab (GPU)
Runtime > Change runtime type > **T4 GPU**. Run the cells in order. Put `processed.zip` (made by Andrea) in `MyDrive/playmatch/` first.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/playmatch'

In [ ]:
!git clone https://github.com/LeilaAlk/playmatch.git
%cd /content/playmatch
!pip install -q -r requirements.txt

In [ ]:
!mkdir -p data && unzip -q -o $DRIVE/processed.zip -d data/
!ls data/processed | head -30   # must list train.parquet, items_bge.npy, nlgcl_items.npy, semantic_ids.parquet, query_sources.parquet

## 1. Requests (Julia)
Test + validation first (about 7 000, 30 to 60 min), saved to the Drive. Then training (10 000).
If Colab disconnects, re-run the clone/unzip cells, copy queries.parquet back from the Drive, and continue.

In [ ]:
!python -m src.queries generate --data data/processed --splits test,valid
!cp data/processed/queries.parquet $DRIVE/
!python -m src.queries generate --data data/processed --splits train
!cp data/processed/queries.parquet results/queries_stats.json $DRIVE/

## 2. GenSAR training (Talia)
Quick check first. If the validation loss goes down, run the full training. **Thursday 21h: decision.**

In [ ]:
!python -m src.train_gensar --data data/processed --sample 5000 --epochs 3

In [ ]:
!python -m src.train_gensar --data data/processed --epochs 10
!zip -qr $DRIVE/gensar_best.zip models/gensar_best results/gensar_curve.png results/gensar_history.json

## 3. Ablation: same model without profile tokens

In [ ]:
!python -m src.train_gensar --data data/processed --epochs 10 --no-profile
!zip -qr $DRIVE/gensar_noprofile_best.zip models/gensar_noprofile_best results/gensar_noprofile_curve.png

## 4. Final evaluation (Julia)

In [ ]:
!python -m src.evaluate_model --data data/processed --split test
!zip -qr $DRIVE/results_week3.zip results

In [ ]:
from IPython.display import Image, display
display(Image('results/gensar_curve.png'))
print(open('results/sample_users.md').read())